In [2]:
### Question 1

%pip install ISLP

from ISLP import load_data

default = load_data('Default')
default.head()
##default.shape



Note: you may need to restart the kernel to use updated packages.


,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


In [3]:
# Report dataset dimensions
# Number of observations (n) and number of predictors (p) in the default dataset
n = default.shape[0]
p = default.shape[1] - 1  # subtract 1 if 'default' is the response variable

print('Shape:', default.shape)
print("Number of observations (n):", n)
print("Number of predictors (p):", p)
print('\nColumn names:', default.columns.tolist())
print('\nData types:')
print(default.dtypes)

# Distribution of the default variable
print('\nDistribution of default:')
print(default['default'].value_counts())

Shape: (10000, 4)
Number of observations (n): 10000
Number of predictors (p): 3

Column names: ['default', 'student', 'balance', 'income']

Data types:
default    category
student    category
balance     float64
income      float64
dtype: object

Distribution of default:
default
No     9667
Yes     333
Name: count, dtype: int64


In [4]:
# Create a new column "bin_default" with 1 for 'Yes' and 0 for 'No' in the 'default' column and teh same for student
default['bin_default'] = default['default'].map({'Yes': 1, 'No': 0})

default['bin_student'] = default['student'].map({'Yes': 1, 'No': 0})



In [5]:
%pip install statsmodels

import statsmodels.api as sm

# Convert bin_default and bin_student to numeric if needed
y = default['bin_default'].astype(int)
X = default[['balance', 'income', 'bin_student']].astype(float)
X = sm.add_constant(X)  # add intercept

logit_model = sm.Logit(y, X)
result = logit_model.fit()
print(result.summary())




Note: you may need to restart the kernel to use updated packages.
Optimization terminated successfully.
         Current function value: 0.078577
         Iterations 10
                           Logit Regression Results                           
Dep. Variable:            bin_default   No. Observations:                10000
Model:                          Logit   Df Residuals:                     9996
Method:                           MLE   Df Model:                            3
Date:                Fri, 26 Sep 2025   Pseudo R-squ.:                  0.4619
Time:                        11:19:49   Log-Likelihood:                -785.77
converged:                       True   LL-Null:                       -1460.3
Covariance Type:            nonrobust   LLR p-value:                3.257e-292
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const         -10.8690      0.492    -2

In [ ]:
from sklearn.model_selection import train_test_split
### question 2 
# Split the data into training (70%) and testing (30%) sets
X_train, X_test, y_train, y_test = train_test_split(
	X, y, test_size=0.3, random_state=42
)




In [7]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pandas as pd

# Use only 'income' and 'balance' as predictors
X_train_reduced = X_train[['income', 'balance']]
X_test_reduced = X_test[['income', 'balance']]

# Fit LDA model
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_reduced, y_train)

# Class means for each predictor
means = pd.DataFrame(lda.means_, columns=X_train_reduced.columns)
means.index = ['No Default', 'Default']
print('Class means for each predictor:')
print(means)

# Prior probabilities for each class
priors = lda.priors_
print('\nPrior probabilities for each class:')
for label, prior in zip(['No Default', 'Default'], priors):
    print(f'{label}: {prior:.4f}')

# Explanation
print("\nExplanation:")
print("- The class means show the average value of each predictor (balance, income) for each class (default/no default).")
print("- The prior probabilities show the proportion of each class in the training data, i.e., the probability that a randomly chosen observation belongs to each class before seeing any predictors.")

Class means for each predictor:
                  income      balance
No Default  33681.793667   802.158374
Default     31570.357690  1768.165821

Prior probabilities for each class:
No Default: 0.9659
Default: 0.0341

Explanation:
- The class means show the average value of each predictor (balance, income) for each class (default/no default).
- The prior probabilities show the proportion of each class in the training data, i.e., the probability that a randomly chosen observation belongs to each class before seeing any predictors.


In [8]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
import pandas as pd

# Use only 'income' and 'balance' as predictors
X_train_reduced = X_train[['income', 'balance']]
X_test_reduced = X_test[['income', 'balance']]

# Fit QDA model
qda = QuadraticDiscriminantAnalysis()
qda.fit(X_train_reduced, y_train)

# Class means for each predictor
means_qda = pd.DataFrame(qda.means_, columns=X_train_reduced.columns)
means_qda.index = ['No Default', 'Default']
print('Class means for each predictor (QDA):')
print(means_qda)

# Prior probabilities for each class
priors_qda = qda.priors_
print('\nPrior probabilities for each class (QDA):')
for label, prior in zip(['No Default', 'Default'], priors_qda):
    print(f'{label}: {prior:.4f}')

# Explanation
print("\nExplanation:")
print("- QDA allows each class to have its own covariance matrix, so the decision boundary is quadratic (curved) rather than linear.")
print("- The class means and priors are interpreted the same way as in LDA.")

Class means for each predictor (QDA):
                  income      balance
No Default  33681.793667   802.158374
Default     31570.357690  1768.165821

Prior probabilities for each class (QDA):
No Default: 0.9659
Default: 0.0341

Explanation:
- QDA allows each class to have its own covariance matrix, so the decision boundary is quadratic (curved) rather than linear.
- The class means and priors are interpreted the same way as in LDA.


In [9]:
from sklearn.metrics import confusion_matrix, accuracy_score

# LDA predictions and evaluation
lda_pred = lda.predict(X_test_reduced)
lda_cm = confusion_matrix(y_test, lda_pred)
lda_acc = accuracy_score(y_test, lda_pred)
print('LDA Confusion Matrix:')
print(lda_cm)
print(f'LDA Test Accuracy: {lda_acc:.4f}\n')

# QDA predictions and evaluation
qda_pred = qda.predict(X_test_reduced)
qda_cm = confusion_matrix(y_test, qda_pred)
qda_acc = accuracy_score(y_test, qda_pred)
print('QDA Confusion Matrix:')
print(qda_cm)
print(f'QDA Test Accuracy: {qda_acc:.4f}')

LDA Confusion Matrix:
[[2900    6]
 [  75   19]]
LDA Test Accuracy: 0.9730

QDA Confusion Matrix:
[[2898    8]
 [  70   24]]
QDA Test Accuracy: 0.9740


In [11]:
from sklearn.naive_bayes import GaussianNB


### Question 3 
# Fit Naive Bayes model using only 'income' and 'balance'
nb = GaussianNB()
nb.fit(X_train_reduced, y_train)

# Predict on test set
nb_pred = nb.predict(X_test_reduced)
nb_cm = confusion_matrix(y_test, nb_pred)
nb_acc = accuracy_score(y_test, nb_pred)
print('Naive Bayes Confusion Matrix:')
print(nb_cm)
print(f'Naive Bayes Test Accuracy: {nb_acc:.4f}\n')

# Compare with LDA and QDA (already printed above)
print('LDA Test Accuracy (for comparison):', f'{lda_acc:.4f}')
print('QDA Test Accuracy (for comparison):', f'{qda_acc:.4f}')

# Predict probability for a specific customer
customer = [[40000, 2000]]
prob_default = nb.predict_proba(customer)[0,1]
print(f'Predicted probability of default for income=40000 and balance=2000: {prob_default:.4f}')

Naive Bayes Confusion Matrix:
[[2893   13]
 [  75   19]]
Naive Bayes Test Accuracy: 0.9707

LDA Test Accuracy (for comparison): 0.9730
QDA Test Accuracy (for comparison): 0.9740
Predicted probability of default for income=40000 and balance=2000: 0.5108


c:\Users\Owen Hirsch\anaconda3\envs\NewKernel\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GaussianNB was fitted with feature names
  warnings.warn(


In [ ]:
from sklearn.preprocessing import StandardScaler

### Question 4

# Apply feature scaling to income and balance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_reduced)
X_test_scaled = scaler.transform(X_test_reduced)

# Fit Naive Bayes model on scaled data
nb_scaled = GaussianNB()
nb_scaled.fit(X_train_scaled, y_train)

# Predict on test set
nb_scaled_pred = nb_scaled.predict(X_test_scaled)
nb_scaled_cm = confusion_matrix(y_test, nb_scaled_pred)
nb_scaled_acc = accuracy_score(y_test, nb_scaled_pred)
print('Naive Bayes (Scaled) Confusion Matrix:')
print(nb_scaled_cm)
print(f'Naive Bayes (Scaled) Test Accuracy: {nb_scaled_acc:.4f}\n')

# Predict probability for a specific customer (scaled)
customer_scaled = scaler.transform([[40000, 2000]])
prob_default_scaled = nb_scaled.predict_proba(customer_scaled)[0,1]
print(f'Predicted probability of default for income=40000 and balance=2000 (scaled): {prob_default_scaled:.4f}')

Naive Bayes (Scaled) Confusion Matrix:
[[2893   13]
 [  75   19]]
Naive Bayes (Scaled) Test Accuracy: 0.9707

Predicted probability of default for income=40000 and balance=2000 (scaled): 0.5108


c:\Users\Owen Hirsch\anaconda3\envs\NewKernel\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd

k_values = [1, 3, 5, 10]
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    acc = knn.score(X_test_scaled, y_test)
    accuracies.append(acc)

# Create summary table
summary = pd.DataFrame({'K': k_values, 'Test Accuracy': accuracies})
print(summary)

### the most accurate is k = 10. That makes sense as it is making predictions based on the 10 nearest neighbors. This model is less flexible and so is going to be a better predictor in this case.
### the reason why the lower K values have lower accuracies is probably because there is a lot of overlap between those that default and those that do not. So if you are only looking at the nearest neighbor you are more likely to pick one that is in the wrong class - increasing K means that you are looking for how the nearest group is moving.



    K  Test Accuracy
0   1       0.953667
1   3       0.966333
2   5       0.969667
3  10       0.972000


In [17]:
# Summary table comparing all methods
from sklearn.linear_model import LogisticRegression

# Refit Logistic Regression on training data (using scaled features for fair comparison)
logreg = LogisticRegression()
logreg.fit(X_train_scaled, y_train)
logreg_acc = logreg.score(X_test_scaled, y_test)

# Best KNN accuracy (already computed above)
best_knn_acc = max(accuracies)
best_k = k_values[accuracies.index(best_knn_acc)]

summary_all = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'LDA',
        'QDA',
        'Naive Bayes',
        f'KNN (K={best_k})'
    ],
    'Test Accuracy': [
        logreg_acc,
        lda_acc,
        qda_acc,
        nb_scaled_acc,
        best_knn_acc
    ]
})
print(summary_all)

# Display all confusion matrices for comparison
from sklearn.metrics import confusion_matrix

# Logistic Regression predictions and confusion matrix
logreg_pred = logreg.predict(X_test_scaled)
logreg_cm = confusion_matrix(y_test, logreg_pred)

# Best KNN predictions and confusion matrix
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_scaled, y_train)
knn_pred = knn_best.predict(X_test_scaled)
knn_cm = confusion_matrix(y_test, knn_pred)

print('Logistic Regression Confusion Matrix:')
print(logreg_cm)
print('\nLDA Confusion Matrix:')
print(lda_cm)
print('\nQDA Confusion Matrix:')
print(qda_cm)
print('\nNaive Bayes (Scaled) Confusion Matrix:')
print(nb_scaled_cm)
print(f'\nKNN (K={best_k}) Confusion Matrix:')
print(knn_cm)

                 Model  Test Accuracy
0  Logistic Regression       0.973333
1                  LDA       0.973000
2                  QDA       0.974000
3          Naive Bayes       0.970667
4           KNN (K=10)       0.972000
Logistic Regression Confusion Matrix:
[[2896   10]
 [  70   24]]

LDA Confusion Matrix:
[[2900    6]
 [  75   19]]

QDA Confusion Matrix:
[[2898    8]
 [  70   24]]

Naive Bayes (Scaled) Confusion Matrix:
[[2893   13]
 [  75   19]]

KNN (K=10) Confusion Matrix:
[[2894   12]
 [  72   22]]


In [ ]:
### the QDA matrix is the best if a false Negative is 10x worse than a false positive. It has the lowest number of false negatives compared to the other models with Logit coming in as a close second.
### all methods miss far more defaults than they correctly identify.


In [19]:
# QDA predictions and evaluation with threshold 0.3
qda_proba = qda.predict_proba(X_test_reduced)[:,1]
qda_pred_03 = (qda_proba > 0.3).astype(int)
qda_cm_03 = confusion_matrix(y_test, qda_pred_03)
qda_acc_03 = accuracy_score(y_test, qda_pred_03)
print('QDA Confusion Matrix (threshold=0.3):')
print(qda_cm_03)
print(f'QDA Test Accuracy (threshold=0.3): {qda_acc_03:.4f}')


### if we remake the propability it makes it far more accurate given the cost of a false negative is 10x worse than a false positive. False positives increase significantly, but the decrease in false negatives out ways it.

QDA Confusion Matrix (threshold=0.3):
[[2858   48]
 [  53   41]]
QDA Test Accuracy (threshold=0.3): 0.9663
